# E2E Austria AML run (Sentinel-1 RTC)

The Sentinel-1 version of [`e2e_austria_aml.ipynb`](./e2e_austria_aml.ipynb). It runs over ONE ~5 km
cell (`s2grid=4772924`, 43 labelled fields), June 2018, ascending passes only. A download copies
whole ~3.7 GB scenes, so `AT_ROI` over April–September would be ~388 GB of Sentinel-1.

> **Run [`build_images.ipynb`](./build_images.ipynb) first.** This notebook assumes
> `rise-eo-aml-env` and `rise-eo-infer-sklearn` are already registered **and finished building** in ACR.

## What lives where

```
notebooks/
  demo_model/       YOUR model only: my_adapter_s1.py + rf_s1.joblib
  demo_bundle_s1/      written by bundle.save below
  demo_verify_adapter/<RUN>/  written by rise_eo.verify_adapter below: one real cube,
                    output.tif, grids.geojson, _result.json -- open the tif in QGIS
  demo_training_data_s1/  written by create_training_data below
  (the model registry is NOT here -- it lives on blob at $AZ_ROOT/model_registry:
   one folder per model name, one immutable v<N> per deployed bundle, plus _aliases.json)
```

`bundle.save` reads only `demo_model/`. There are no image build contexts in this repo:
`rise_eo.image.ImageDefinition` renders the Dockerfile and builds the wheel into a temp
directory at build time, and `verify_image` catches a stale image by reading the image
registry's record of what went into it rather than a wheel left behind on disk.
## The shape of the whole thing

```
1. create training data   rise_eo.create_training_data                    -> section 1
2. create features        rise_eo.bands.modify + apply_features           -> section 2.1
3. train the model        your own sklearn code, not rise-eo's            -> section 2.2
4. deploy the model       five steps, each one a gate                 -> section 3
5. run inference          rise_eo.download, then rise_eo.run_inference        -> section 4
```

Two steps hide inside those five, and both are easy to forget:

- **Step 5 begins with a download.** `run_inference` never fetches imagery — it reads a catalog
  and refuses to go to the source. That is deliberate: a fan-out of 299 nodes must not each
  decide to hit CDSE. So inference over a new ROI means `rise_eo.download` first, then
  `rise_eo.run_inference`. Step 1 has a download too, but it is folded inside
  `create_training_data(download=True)`, which is why you do not call it yourself there.
- **Before any of it: the images.** `00_build_images.ipynb` registers `rise-eo-aml-env` (the pipeline
  legs) and `rise-eo-infer-sklearn` (inference). Once per dependency *family*, never per model —
  retraining does not touch them. Both notebooks call `rise_eo.aml.ensure_environment` against the
  same image registry, so whichever you run second reuses what the first one built.


## 0. Setup

In [ ]:
import datetime, geopandas as gpd, rise_eo

In [ ]:
# Your Azure settings live in ~/.config/rise-eo/config.toml (outside this repo) and are read by
# rise_eo.config.load(). Run `rise-eo init` once to fill them in
import os
import pathlib

import rise_eo.aml
from rise_eo.image import ImageDefinition

REPO = pathlib.Path.cwd().parent
NOTEBOOKS = REPO / "notebooks"

cfg = rise_eo.config.load()

# AZ_ROOT is the root folderpath on azure blob storage where all the intermediate files
# and outputs are created.
AZ_ROOT = "abfss://data@strisewesteurope.dfs.core.windows.net/nsasiraj/demo-runs"

# For running things on azure cluster, you need to create images. The images required to run
# this notebook have been created in build_images.ipynb
# Images created are tracked using the image_registry
IMAGE_REGISTRY = cfg.image_registry
assert IMAGE_REGISTRY, "set image_registry: `rise-eo init --set image_registry=abfss://…/image_registry`"

# BASE image is the image used for steps that do not depend on the model dependent libraries.
# Pure rise-eo dependency - create_training_data, download
# This image has already been created in the build_images.ipynb notebook. Here we just check if
# the image is present and link to it.
BASE = ImageDefinition(
    name="rise-eo-aml-env",
    rise_eo="git+https://github.com/nasaharvest/rise-eo@v0.3.0",
    extras=("azure", "mpc", "local"),
)
# INFER image is used in the steps that use the model. For our example we have two additional
# dependencies on top of rise-eo - sklearn and joblib
INFER = BASE.derive(name="rise-eo-infer-sklearn", extra_pip=("scikit-learn", "joblib"))

# Linking to the images
_env = rise_eo.aml.ensure_environment(
    BASE, registry=IMAGE_REGISTRY,
    resource_group=cfg.resource_group, workspace=cfg.workspace,
    storage="azure",   # the image registry is on blob, same as the model registry below
)
_infer = rise_eo.aml.ensure_environment(
    INFER, registry=IMAGE_REGISTRY,
    resource_group=cfg.resource_group, workspace=cfg.workspace,
    storage="azure",   # the image registry is on blob, same as the model registry below
)

# We are ensuring that the images were already created in the build_images.ipynb notebook
# and not being created anew in this notebook.
for _r in (_env, _infer):
    assert _r.reused, (
        f"{_r.ref} was just built -- open {_r.build_url} , wait for `Build status: Succeeded`, "
        "then re-run this cell. It will say `reusing` and carry on."
    )

AZ_ENV_NAME, AZ_ENV_VERSION = _env.name, _env.version
AZ_INFER_ENV_NAME, AZ_INFER_ENV_VERSION = _infer.name, _infer.version
print(f"images: {_env.ref} / {_infer.ref}   (resolved from the registry, not pasted)")

# Model registry is how we keep track of the different models that have been trained and
# deployed on azure.
# A model is identified by two names - the main name and the version or the alias of the model.
# For this exampled the name of the model is `crop-rf-s1` and teh "alias" is `demo-eurocrops-at`
#
# To use the model trained by this notebook, we would identify the model as follows:
#   "crop-rf-s1@demo-eurocrops-at" 
REGISTRY   = cfg.model_registry
assert REGISTRY, "set model_registry: `rise-eo init --set model_registry=abfss://…/model_registry`"
MODEL_NAME = "crop-rf-s1"
MODEL_ALIAS = "demo-eurocrops-at"

# Test geometries
SHAPEFILES = NOTEBOOKS / "shapefiles"

# RESUMING A RUN. Each run is identified by the name of the folder. Steps that have already been
# gets skipped if outputs already exist.

# RESUME_RUN pins ROOT, where the IMAGERY + catalog live:
#     RESUME_RUN = None            -> new archive; the download leg re-downloads
#     RESUME_RUN = "demo-2026..."  -> re-enter that archive; [download] skips
RESUME_RUN = None   # <- paste a previous RUN here to re-enter it
RUN  = RESUME_RUN or (
    "demo-" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
ROOT = f"{AZ_ROOT}/{RUN}"
print("RUN =", RUN, "(resumed)" if RESUME_RUN else "(fresh)")

# For satellite images and training data we are maintaining a static folderpath which is
# independent of the runs. This is so that different runs don't re-download the satellite
# images and regenerate the training data 
TRAIN_RUN_FOLDERPATH = f"{ROOT}/runs/train"
IMAGERY_FOLDERPATH = f"{ROOT}/imagery"

# The archive holds one folder per collection, and the catalog sits inside it:
# {IMAGERY_FOLDERPATH}/sentinel-1-rtc/catalog.parquet
CATALOG_FILEPATH = rise_eo.archive_catalog_filepath(IMAGERY_FOLDERPATH, "sentinel-1-rtc")

runner_kwargs = dict(
    cluster = cfg.cluster,
    environment = f"{AZ_ENV_NAME}:{AZ_ENV_VERSION}",
    root = ROOT,
    identity_client_id = cfg.uami_client_id,   # the NODES' managed identity, not your login
    subscription_id = cfg.subscription_id,
    resource_group_name = cfg.resource_group,
    workspace_name = cfg.workspace,
    poll_interval_seconds = 10,
)

In [ ]:
print('ROOT =', ROOT)

In [ ]:
fields = gpd.read_file(SHAPEFILES / "AT_2018_TRAIN.geojson")   # 900 fields, cols: fid, crop

# Keep only the fields inside the one cell this run covers. download=True downloads for the
# label polygons, so all 900 fields would pull whole S1 scenes across Austria.
_cell = gpd.read_file(SHAPEFILES / "s2grid=4772924.geojson")
fields = fields.to_crs(_cell.crs)
fields = fields[fields.intersects(_cell.geometry.iloc[0])]   # 43 fields

print('shape:', fields.shape, '\n')
print('columns:', fields.columns.tolist(), '\n')
print(fields['crop'].value_counts(), '\n')
fields.head()

## 1. Create training data (Sentinel-1 RTC)

For a given:
- polygons with labels (labels are optional)
- `startdate`, `enddate`, `mosaic_days`
- `bands`
- `collection = "sentinel-1-rtc"`. SAR has no cloud mask, so no mask step runs.
- `properties_filter = {"sat:orbit_state": "ascending"}`. Required for sentinel-1-rtc: it refuses to mosaic ascending and descending passes together.
- source
- `export_folderpath`

The function creates arrays containing pixel-wise timeseries of bands and downloads to `export_folderpath`:
- data.npy            : Array containing timeseries of the bands
- coords.npy          : Array containing the lat-lon in EPSG:4326 of the pixels
- ids.npy             : Array containing the `id` of each entry in data.npy to map back to the polygons file
- labels.npy          : Array containing the label to each entry in data.npy
- metadata.pickle.npy : Contains information of the order of the bands in data.npy and more helper informations.

- features.npy, feature_ids.npy, feature_labels.npy -> After `"median_per_id"` aggregation


If data is already downloaded, no runs are initiated on the azure cluster.

If parameters are changed - (bands, mosaic scheme, collection) - training data creation is redone.

If there are fields with no-imagery, they are recorded in {TRAIN_RUN_FOLDERPATH}/_manifest.json

`overwrite= `forces a leg past its skip:
- False (default) skip what's done | "datacubes" rebuild cubes (and re-flatten)
- "flatten" keep cubes, redo the flatten | True both  
- "datacubes" and True also refresh every per-field catalog slice and clear the known-empty record -- that is the escape hatch after re-ingesting imagery.

In [ ]:
td = rise_eo.create_training_data(
    label_polygons = fields,                        # in-memory gdf -> auto-staged to blob
    catalog_filepath = CATALOG_FILEPATH,           # images are downloaded to {IMAGERY_FOLDERPATH}/sentinel-1-rtc/
    startdate = datetime.datetime(2018, 6, 1),
    enddate = datetime.datetime(2018, 7, 1),
    mosaic_days = 10,
    bands = ["vv", "vh"],
    collection = "sentinel-1-rtc", properties_filter = {"sat:orbit_state": "ascending"},
    id_col = "fid", label_col = "crop",
    export_folderpath = "./demo_training_data_s1",    # LOCAL landing dir for the compact array
    aggregate = "median_per_id",                   # one row per field, not per pixel
    source = "mpc", download = True,
    overwrite = False,                             # skip whatever is already done
    max_tiles = 250,                               # no max_cloudcover: S1 has no cloud cover
    runner = "aml", runner_kwargs = runner_kwargs,
    run_folderpath = TRAIN_RUN_FOLDERPATH,
)

## 2. Model training

The training data that was downloaded currently contains raw bands.

1. create the features we want to train our model with. In our case none: `vv` and `vh` go to the model as they are, after masking nodata and filling gaps. NDVI and SAVI are optical indices; rise-eo has no SAR index.
2. train a random-forest model.

### 2.1 Create features

In [ ]:
import numpy as np
from rise_eo.bands import modify
from rise_eo.model.features import apply_features   # the one place feature_sequence is applied

In [ ]:
d = td.load()
raw, ids, y = d["features"], d["feature_ids"], np.asarray(d["feature_labels"])  # (fields, 3, 2)
band_indices = {b: i for i, b in enumerate(td.bands)}                           # vv, vh

# no features to compute: mask nodata, fill the gaps, keep vv and vh
SEQ = [
    (modify.mask_invalid_and_interpolate, {}),
]

feats5d, feat_bi = apply_features(modify.expand_flattened(raw.astype(float)),
                                  dict(band_indices), feature_sequence=SEQ)
feats = np.squeeze(feats5d, axis=(2, 3))         # (fields, 3, 2)

print(feats.shape, sorted(feat_bi, key=feat_bi.get))

### 2.2 Train

In [ ]:
import joblib, os
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

In [ ]:
X = feats.reshape(len(feats), -1)
keep = ~np.isnan(X).any(axis=1)
X, yk = X[keep], y[keep]
le = LabelEncoder(); yy = le.fit_transform(yk)

clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
print("field-wise CV:", cross_val_score(clf, X, yy, cv=5).mean())   # rows are fields → honest
clf.fit(X, yy)

outdir = "./demo_model"
os.makedirs(outdir, exist_ok=True)
model_fp = os.path.join(outdir, "rf_s1.joblib")
joblib.dump((clf, le), model_fp)

In [ ]:
dict(zip(le.classes_, range(len(le.classes_))))

## 3. Deploying the model

**"Deploy" is five steps, not one.** Each is a gate: it either passes, or it tells you which of
the five is wrong. That is the whole reason they are separate calls rather than one button.

| | step | what it proves | where it runs |
|---|---|---|---|
| 3.1 | write the model adapter | nothing yet — this is code you write | — |
| 3.2 | `rise_eo.verify_adapter` | your adapter runs on a **real** datacube, and the output looks right in QGIS | one AML node, then your machine |
| 3.3 | `bundle.save` | model + code + requirements travel as **one unit** | your machine |
| 3.4 | `rise_eo.model.verify_image` | the inference **image** can load and run that bundle | one AML node |
| 3.5 | `rise_eo.deploy` | the verified bundle gets a **name** in the registry | your machine + blob |

**A prerequisite that is not in this notebook.** Step 3.4 needs the inference image to already
exist — `rise-eo-infer-sklearn`, built and registered by
[`00_build_images.ipynb`](./00_build_images.ipynb). It is generic per dependency *family*, so
you build it once and every sklearn model you ever train reuses it. Retraining never touches it.

**One ordering rule.** 3.3 -> 3.4 -> 3.5 run in that order, and you must not re-run 3.3 in
between. `deploy` accepts 3.4's verdict (`verified=vres`) only while the bundle content that
verdict fingerprinted still matches what you are deploying. Re-bundling invalidates the proof,
and `deploy` refuses rather than registering something the image was never run against.


### 3.1 Creating a model adapter

The adapter is how rise-eo's inference pipeline uses the model

For this example the adapter is written to ./demo_model/my_adapter_s1.py

```python
from rise_eo.bands import modify
from rise_eo.model.adapter import BaseModelAdapter

class CropRFS1(BaseModelAdapter):
    required_bands = ["vv", "vh"]          # raw bands SEQ consumes
    n_timestamps = 0                       # set per-instance at bundle time
    output_dtype = "uint8"
    output_nodata = 255
    output_band_names = ["crop_class"]
    feature_sequence = [
        (modify.mask_invalid_and_interpolate, {}),
    ]

    def load(self):
        import joblib
        self.clf, self.le = joblib.load(self.artifacts["model"])

    def predict(self, X_chunk):
        return self.clf.predict(X_chunk).astype("uint8")
```

Currently there is no mechanism implemented to check if the processing steps used to generate
the training data is the same that is defined in the adapter. It is the modeller's responsibility
to ensure that.

The next cell, `rise_eo.verify_adapter`, runs this adapter on one real datacube before you trust
it at scale.

In [ ]:
# Manually checking if the processing steps in training data generation matches
# the model adapter
import sys; sys.path.insert(0, "./demo_model")
from my_adapter_s1 import CropRFS1

assert CropRFS1.feature_sequence == SEQ, (
    "the adapter's feature_sequence has drifted from the SEQ you trained on:\n"
    f"  trained on : {SEQ}\n"
    f"  adapter    : {CropRFS1.feature_sequence}"
)
print("features match:", [fn.__name__ for fn, _ in SEQ])


### 3.2 Verify the adapter

Verify the adapter by running the model on one datacube.

`rise_eo.verify_adapter` builds **one** ~5km grid cell datacube on AML, downloads it, and runs the model via the adapter locally.
The output of the adapter is `output.tif`, which is to be opened on QGIS to verify if model worked as expected.

We are checking:
1. whether the image - which was created in build_images.ipynb - works against the model.
2. if the adapter, as written, works - because the user needs write the code to interact with the datacube themselves.


In [ ]:
# demo_model/ holds exactly two things: the adapter source and the trained model.
from rise_eo.model import bundle

In [ ]:
adapter = CropRFS1()
adapter.n_timestamps = rise_eo.compute_n_timestamps(
    datetime.datetime(2018, 6, 1), datetime.datetime(2018, 7, 1), 10)   # 3
adapter.artifacts = {"model": model_fp}

# Model adapter is verified by running on a datacube. The datacube gets created on AML
# and downloaded to local. If the local datacube already exists, the datacube creation
# step is skipped.
report = rise_eo.verify_adapter(
    adapter,                                          # the LIVE adapter -> auto-bundled
    roi              = str(SHAPEFILES / "s2grid=4772924.geojson"),
    catalog_filepath = CATALOG_FILEPATH,
    startdate        = datetime.datetime(2018, 6, 1),
    enddate          = datetime.datetime(2018, 7, 1),
    mosaic_days      = 10,
    bands            = ["vv", "vh"],                  # what run_inference builds, not required_bands
    collection       = "sentinel-1-rtc",
    properties_filter = {"sat:orbit_state": "ascending"},
    cell             = None,                          # deterministic pick; prints the id + why
    export_folderpath = f"./demo_verify_adapter/{RUN}",
    storage = "azure", runner = "aml", runner_kwargs = runner_kwargs,
)

# pass: False is a finding ABOUT THE ADAPTER (a T mismatch, a wrong output dtype) and comes back
# in the dict; only a bad CALL raises. Either way the verdict is on disk as _result.json.
assert report["pass"], report["error"]

m = report["metrics"]
print("cell           ", m["cell"])
print("cube shape     ", m["cube_shape"], "T =", m["cube_t"], "(adapter wants", m["adapter_n_timestamps"], ")")
print("bands in cube  ", m["cube_bands"])
print("after features ", m["post_feature_sequence_bands"], "| required:", m["required_bands"])
print("output         ", m["output_dtype"], "range", (m["output_value_min"], m["output_value_max"]),
      "| nodata frac", m["output_nodata_fraction"])
print()
print("OPEN THESE IN QGIS -- the verdict above is assistive, the raster is the deliverable:")
print("  ", m["output_filepath"])
print("  ", m["grids_filepath"])

# 2m 51.2s - 6m 34.6s (node startup sometimes take 5 minutes)
# 3.3s (pre-ready datacube)

### 3.3 Bundling the model

A model bundle is three things:
1. The model binary (rf_s1.joblib) - DONE
2. The code to run the model (my_adapter_s1.py) - DONE
3. The list of requirements to run the model's code - what we do NOW.

`bundle.save` will put the model binary and the model adapter in a "bundle" directory and creates a bundle.json file containing the requirements to run the model along with other model requirements - bands, operations, data shape - which is how rise-eo knows whether the model would run and what it needs to run the model before we launch a large scale inference run.

In [ ]:
bundle_dir = bundle.save(
    adapter = adapter,
    artifacts = {"model": model_fp},
    dst = "./demo_bundle_s1",
    code = ["./demo_model/my_adapter_s1.py"],
    requirements = ["scikit-learn>=1.5", "joblib"],
)

spec = bundle.read_spec(bundle_dir)
print(spec["adapter"])          # my_adapter_s1:CropRFS1
print(spec["fsd_bundle_version"], spec["code"])   # 2  {'root': 'code', 'files': ['my_adapter_s1.py']}

### 3.4 Verify inference image

Verifying whether the image chosen works for the model.

If this fails then the fix is either making corrections in the adapter, or creating a new image - latter would involve re-running "build_images.ipynb" with custom configuration.

In [ ]:
from rise_eo.model import verify_image

vres = verify_image(
    bundle_dir,
    environment    = f"{AZ_INFER_ENV_NAME}:{AZ_INFER_ENV_VERSION}",
    runner         = "aml",
    runner_kwargs  = runner_kwargs,
)
assert vres["pass"], vres               # don't spend an inference fan-out on an unverified image
vres["metrics"]

# 3m 24.4s

### 3.5 Deploy the model to the registry

**This is where a verified model gets a NAME.** `rise_eo.deploy` publishes `./demo_bundle_s1` into
MODEL REGISTRY, which is a folder on the azure blob. The model gets assigned a version id
which is returned by `rise_eo.deploy`, example `crop-rf-s1:1`.

`deploy` requires the model to be verified against an image (step 3.4).

In [ ]:
print(MODEL_NAME, '@', MODEL_ALIAS)

In [ ]:
MODEL_REF = rise_eo.deploy(
    bundle_dir,
    name        = MODEL_NAME,               # "crop-rf-s1" -- no "/", ":" or "@" (a ref must parse)
    registry    = REGISTRY,                 # on BLOB -- see the config cell
    storage     = "azure",                  # forbid the anonymous fallback for that write
    environment = f"{AZ_INFER_ENV_NAME}:{AZ_INFER_ENV_VERSION}",   # SAME string verify_image got
    verified    = vres,                     # reuse the gate above; stale/mismatched -> refused
    alias       = MODEL_ALIAS,              # publish + repoint in one call
)
print(MODEL_REF)                            # crop-rf-s1:1  <- a version PIN, always exact

# What landed. Nothing the registry writes contains an absolute path, so the whole folder
# can be moved or copied elsewhere and every name still resolves to the same version.
from rise_eo.model import registry as _registry

resolved = _registry.resolve(f"{MODEL_NAME}@{MODEL_ALIAS}", REGISTRY)
print(resolved)                              # Resolved(name=..., version=1, path=...)

rec = _registry.read_deploy_record(resolved.path)   # _deploy.json -- the bundle<->image binding
print("environment :", rec["environment"])   # what this version was PROVEN to run on
print("digest      :", rec["digest"])        # what content that proof covered
print("deployed_at :", rec["deployed_at"])

## 4. Run inference

Step 5 of the map at the top, and the only step that is two calls: `run_inference` reads a
catalog and never fetches imagery itself, so a new ROI needs `rise_eo.download` first.


### 4.1 Download the imagery inference will read


In [ ]:
rise_eo.download(roi = str(SHAPEFILES / "s2grid=4772924.geojson"),
             dst_folderpath = IMAGERY_FOLDERPATH,   # catalog lands at CATALOG_FILEPATH
             startdate = datetime.datetime(2018, 6, 1),
             enddate = datetime.datetime(2018, 7, 1),
             bands = ["vv", "vh"],
             source = "mpc", collection = "sentinel-1-rtc",
             properties_filter = {"sat:orbit_state": "ascending"},
             max_tiles = 250,
             runner = "aml", runner_kwargs = runner_kwargs)

### 4.2 Run inference off the deployed NAME


In [ ]:
infer_kwargs = dict(runner_kwargs)
infer_kwargs["environment"] = f"{AZ_INFER_ENV_NAME}:{AZ_INFER_ENV_VERSION}"

In [ ]:
result = rise_eo.run_inference(
    model    = MODEL_REF,
    registry = REGISTRY,
    output_folderpath = f"{ROOT}/model_outputs/s2grid=4772924",
    roi = str(SHAPEFILES / "s2grid=4772924.geojson"),
    catalog_filepath = CATALOG_FILEPATH,
    startdate = datetime.datetime(2018, 6, 1),
    enddate = datetime.datetime(2018, 7, 1),
    mosaic_days = 10,
    bands = ["vv", "vh"],
    collection = "sentinel-1-rtc", properties_filter = {"sat:orbit_state": "ascending"},
    merge = "reproject", overwrite = False,
    storage = "azure", runner = "aml", runner_kwargs = infer_kwargs,
)
print(len(result.output_filepaths), result.merged_filepath, result.stac_catalog_filepath)

# str(SHAPEFILES / "s2grid=4772924.geojson") -> 1 cell